# 07 — Thermal Comfort (UTCI) and Anthropogenic Heat Flux

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/07_thermal_comfort_anthropogenic_heat.en.ipynb)

**Learning objective**: compute the Universal Thermal Climate Index (UTCI) — the internationally standardized human-biometeorological heat-stress index used in heat-health warning systems — and estimate anthropogenic heat flux (Q_F, the waste heat released by buildings, traffic, and people), both driven directly by Local Climate Zone class. By the end you will understand *why* two neighborhoods reporting the identical air temperature can pose very different heat-stress risk to people, and why compact urban LCZs are also net producers of extra heat that the city itself has to absorb.

## Why LCZ class matters for human thermal comfort and anthropogenic heat

The Stewart & Oke (2012) LCZ scheme was built around measurable physical properties of urban form — building surface fraction, impervious surface fraction, height-to-width ratio, and **sky view factor (SVF)** among them. Two of these properties feed directly into human-scale climate hazards that a simple air-temperature map cannot capture on its own.

**Thermal comfort and UTCI.** The Universal Thermal Climate Index (Fiala et al. 2012, ISO 7243) is the index most heat-health early-warning systems worldwide are built around, because it combines the four variables that actually determine how a human body experiences heat: air temperature, wind speed, humidity, and **mean radiant temperature (Tmrt)** — the net radiant heat load from all surrounding surfaces (walls, pavement, sky). Air temperature alone is a poor proxy for heat stress in cities: a person standing in a deep, low-SVF street canyon in a Compact Highrise (LCZ 1) district receives far more reflected and re-emitted longwave radiation from surrounding walls and pavement than someone standing in an open field (LCZ D / class 14), even if a thermometer reports the exact same air temperature at both locations. `lcz_utci` captures this by estimating Tmrt from the LCZ's sky view factor when you don't have a direct Tmrt measurement (which is rare outside dedicated micrometeorological campaigns) — compact, low-SVF LCZs trap more outgoing longwave radiation and push Tmrt (and therefore UTCI) well above air temperature, while open, high-SVF LCZs and natural land covers stay much closer to it. This is exactly the mechanism explored quantitatively in the LCZ4r framework paper (Anjos et al. 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0).

**Anthropogenic heat flux (Q_F).** Anthropogenic heat is the waste heat released directly into the urban energy balance by human activity — building heating/cooling systems, vehicle engines, industrial processes, and the metabolic heat of people themselves. It is a genuine *source term* in the urban surface energy balance, distinct from (and additive to) the radiation-trapping and reduced-evapotranspiration effects that make cities warmer. Q_F correlates strongly with LCZ class because building density, imperviousness, and traffic/industrial activity all vary systematically across the LCZ typology — a Heavy Industry zone (LCZ 10) or Compact Highrise core routinely produces an order of magnitude more waste heat per square metre than a Sparsely Built or vegetated periphery.

## Install LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Get a real LCZ map to work with

We use Vaduz (Liechtenstein) — a small city whose bounding box keeps the download fast. We read the classified raster into a plain NumPy array of LCZ class codes (1–17, 0 = nodata) with `rasterio`; both `lcz_utci` and `lcz_anthropogenic_heat` are pure NumPy functions that operate directly on such arrays (or on scalars/1-D arrays), with no further network access. We also drop the nodata (0) pixels before any per-class statistics — good practice whenever you aggregate by LCZ class, regardless of function.

In [2]:
from LCZ4py import lcz_get_map
import rasterio
import numpy as np

map_path = lcz_get_map(city="Vaduz")
print(map_path)

with rasterio.open(map_path) as src:
    lcz_array = src.read(1).astype(np.int32)

print("Raster shape:", lcz_array.shape)
print("LCZ classes present:", np.unique(lcz_array))

# Drop nodata (0) pixels before feeding class values into lcz_anthropogenic_heat
# -- always mask nodata before per-class analysis, LCZ or otherwise.
lcz_valid = lcz_array[lcz_array > 0]
print("Valid (non-nodata) pixels:", lcz_valid.shape)

06:29:01 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif
Raster shape: (120, 131)
LCZ classes present: [ 0  5  6  8  9 11 12 14 15 17]
Valid (non-nodata) pixels: (2539,)


## Anthropogenic heat flux from real LCZ classes: `lcz_anthropogenic_heat`

`lcz_anthropogenic_heat(lcz_classes, method="simple"|"detailed", params=None, lang="en")` estimates Q_F (W/m²) for every pixel of an LCZ class array and returns an `AnthropogenicHeatResult` with:

- `array` — Q_F per pixel (W/m²), same shape as the input `lcz_classes`,
- `stats` — per-class mean/min/max/pixel-count plus an area-weighted `total_mean`,
- `plot` — a Plotly heatmap (2-D input) or bar chart (1-D input) of Q_F by class.

**`method="simple"`** looks up a literature-derived `(min, max)` Q_F range per LCZ class (e.g. 60–120 W/m² for Compact Highrise, 8–20 W/m² for Sparsely Built, 0–5 W/m² for every natural/water class) and assigns each pixel the midpoint of its class's range. It is fast, transparent, and easy to override via the `params` dict, but it ignores within-class variability — every Compact Highrise pixel gets exactly the same value regardless of how dense or tall that particular block actually is.

**`method="detailed"`** instead derives Q_F from the LCZ's actual morphological parameters (via `lcz_get_parameters`'s lookup table): building surface fraction (BSF) and mean building height (HRE) drive a building-energy term, impervious surface fraction (ISF) drives a traffic term, and a combined built-fraction proxy drives a simplified human-metabolic term. It is more physically grounded (it responds to each class's actual geometry rather than a fixed literature midpoint) but is still a simplified parameterization, not a calibrated bottom-up energy-balance model — treat both methods as order-of-magnitude estimates, with "detailed" preferred when you need the geometry-sensitivity and "simple" preferred when you just need a fast, literature-anchored baseline.

In [3]:
from LCZ4py.local.lcz_thermal import lcz_anthropogenic_heat

qf_simple = lcz_anthropogenic_heat(lcz_valid, method="simple")

print("Q_F array shape:", qf_simple.array.shape)
print("Area-weighted mean Q_F (simple):", round(qf_simple.stats["total_mean"], 2), "W/m2")
for cls, s in sorted(qf_simple.stats["per_class"].items()):
    print(f"  LCZ {cls:>2}: mean={s['mean']:.1f} W/m2  (n={s['count']} px)")

qf_simple.plot

Q_F array shape: (2539,)
Area-weighted mean Q_F (simple): 7.54 W/m2
  LCZ  5: mean=35.0 W/m2  (n=101 px)
  LCZ  6: mean=27.5 W/m2  (n=306 px)
  LCZ  8: mean=25.0 W/m2  (n=22 px)
  LCZ  9: mean=14.0 W/m2  (n=119 px)
  LCZ 11: mean=2.5 W/m2  (n=1260 px)
  LCZ 12: mean=2.5 W/m2  (n=88 px)
  LCZ 14: mean=2.5 W/m2  (n=244 px)
  LCZ 15: mean=2.5 W/m2  (n=389 px)
  LCZ 17: mean=2.5 W/m2  (n=10 px)


**Reading the `simple` output**: `qf_simple.stats["total_mean"]` is the area-weighted average waste-heat flux across the whole Vaduz raster — a small alpine town, so expect a modest value dominated by the low-density residential classes actually present in this footprint. The `per_class` breakdown shows the fixed midpoint Q_F assigned to each class found in the raster; every pixel of a given class gets an identical value here, by construction. The heatmap (or bar chart, if the input was 1-D) makes the spatial contrast between built and natural land cover immediately visible.

In [4]:
qf_detailed = lcz_anthropogenic_heat(lcz_valid, method="detailed")

print("Area-weighted mean Q_F (detailed):", round(qf_detailed.stats["total_mean"], 2), "W/m2")
for cls, s in sorted(qf_detailed.stats["per_class"].items()):
    print(f"  LCZ {cls:>2}: mean={s['mean']:.1f} W/m2  (n={s['count']} px)")

qf_detailed.plot

Area-weighted mean Q_F (detailed): 7.99 W/m2
  LCZ  5: mean=30.8 W/m2  (n=101 px)
  LCZ  6: mean=14.2 W/m2  (n=306 px)
  LCZ  8: mean=19.0 W/m2  (n=22 px)
  LCZ  9: mean=7.1 W/m2  (n=119 px)
  LCZ 11: mean=8.3 W/m2  (n=1260 px)
  LCZ 12: mean=4.9 W/m2  (n=88 px)
  LCZ 14: mean=1.3 W/m2  (n=244 px)
  LCZ 15: mean=1.0 W/m2  (n=389 px)
  LCZ 17: mean=0.9 W/m2  (n=10 px)


**Reading the `detailed` output**: compare `qf_detailed.stats["total_mean"]` and the per-class means against the `simple` run above. Because `detailed` is driven by each class's actual BSF/ISF/mean-height parameters rather than a fixed literature midpoint, the two methods can diverge — especially for classes whose real geometry sits far from the "typical" values the literature range assumes. Neither method is a ground-truth measurement; use `simple` for a quick literature-anchored baseline and `detailed` when you want Q_F to respond to the specific morphology of the LCZ map you are analyzing (e.g. comparing two cities with the same LCZ label but different real building density).

## Thermal stress from a synthetic heatwave scenario: `lcz_utci`

`lcz_utci(air_temp, wind_speed, relative_humidity, mean_radiant_temp=None, *, lc_z=None, output="category"|"index", lang="en")` computes UTCI from the four meteorological inputs and returns a `UTCIResult` with:

- `values` — the numeric UTCI index in °C-equivalent (always returned as a flattened 1-D array, even for a 2-D `lc_z` input — reshape it back with `.reshape(lc_z.shape)` if you need a spatial map),
- `categories` — the thermal-stress category label for each value (same shape as `values`),
- `stats` — mean/min/max/std of the index plus the percentage of pixels falling in each category,
- `plot` — a Plotly figure (histogram, since `values` is always 1-D by the time it is plotted).

If `mean_radiant_temp` is not supplied, the function estimates it from the LCZ sky view factor when `lc_z` is given (`Tmrt ≈ Ta + (1 − SVF) × 15`, a simplified approximation of longwave trapping in urban canyons), or simply assumes `Tmrt = air_temp` if neither is provided — so passing `lc_z` is what actually makes UTCI "LCZ-aware" rather than a plain meteorological index.

**Note on this example**: lcz4r_python does not currently bundle any met station dataset with wind speed and relative humidity (the Berlin CSV used elsewhere in the `local/` series only has air temperature), so the example below is **illustrative/synthetic**: we apply one plausible heatwave-afternoon weather reading uniformly across a small transect of LCZ classes, to isolate and show the effect of urban form (via SVF) on thermal stress when the actual weather is held constant.

In [5]:
from LCZ4py.local.lcz_thermal import lcz_utci
from LCZ4py._internal.lcz_parameters_data import LCZ_NAMES

# Illustrative/synthetic example -- NOT real station data.
# lcz4r_python ships no bundled meteorological record with wind speed and
# relative humidity (the Berlin CSV used elsewhere in local/ only has airT),
# so we construct a small transect of LCZ classes and apply ONE uniform,
# plausible heatwave-afternoon weather reading to all of them -- as if a
# single weather station reported this for the whole city.
lcz_transect = np.array([1, 2, 6, 9, 14, 17])  # compact core -> open -> natural -> water
transect_names = [LCZ_NAMES[c - 1] for c in lcz_transect]

air_temp = 34.0          # degC, typical heatwave afternoon
wind_speed = 2.0         # m/s, light breeze
relative_humidity = 45.0 # %

print(list(zip(lcz_transect.tolist(), transect_names)))

[(1, 'Compact highrise'), (2, 'Compact midrise'), (6, 'Open lowrise'), (9, 'Sparsely built'), (14, 'Low plants'), (17, 'Water')]


### `output="index"` — continuous UTCI values

In [6]:
utci_index = lcz_utci(
    air_temp, wind_speed, relative_humidity,
    lc_z=lcz_transect, output="index",
)

for name, val in zip(transect_names, utci_index.values):
    print(f"  {name:<20s}  UTCI = {val:5.1f} degC")

print("\nSummary stats:", utci_index.stats)
utci_index.plot

  Compact highrise      UTCI =  38.6 degC
  Compact midrise       UTCI =  37.4 degC
  Open lowrise          UTCI =  35.0 degC
  Sparsely built        UTCI =  34.2 degC
  Low plants            UTCI =  33.8 degC
  Water                 UTCI =  33.8 degC

Summary stats: {'mean': 35.449852284648934, 'min': 33.76035245481622, 'max': 38.64015226348531, 'std': 1.8978867601236322, 'category_percentages': {'Extreme cold stress': 0.0, 'Very strong cold stress': 0.0, 'Strong cold stress': 0.0, 'Moderate cold stress': 0.0, 'Slight cold stress': 0.0, 'No thermal stress': 0.0, 'Moderate heat stress': 0.0, 'Strong heat stress': 83.33333333333334, 'Very strong heat stress': 16.666666666666664, 'Extreme heat stress': 0.0}}


**Reading the index output**: every one of the six transect points was given the *exact same* air temperature, wind speed, and humidity — the only thing that differs is the LCZ class, and therefore its sky view factor. Compact Highrise (LCZ 1) and Compact Midrise (LCZ 2) — the lowest-SVF, most enclosed classes in the transect — should show the highest UTCI, because their estimated Tmrt is pushed well above air temperature by longwave trapping. Water (LCZ 17) and Low Plants (LCZ 14) — high-SVF, open classes — should sit closer to air temperature. This is the practical payoff of LCZ-aware UTCI: the *same* weather-station reading translates into meaningfully different real-world heat stress depending on where in the city a person is standing.

### `output="category"` — thermal stress categories

The categories come straight from the `UTCI_CATEGORIES` table in `lcz_thermal.py` — ten bins from *Extreme cold stress* through *No thermal stress* to *Extreme heat stress*, matching the standard UTCI assessment scale used in heat-health guidance.

In [7]:
utci_category = lcz_utci(
    air_temp, wind_speed, relative_humidity,
    lc_z=lcz_transect, output="category",
)

for name, cat in zip(transect_names, utci_category.categories):
    print(f"  {name:<20s}  -> {cat}")

  Compact highrise      -> Very strong heat stress
  Compact midrise       -> Strong heat stress
  Open lowrise          -> Strong heat stress
  Sparsely built        -> Strong heat stress
  Low plants            -> Strong heat stress
  Water                 -> Strong heat stress


**Reading the categories**: under this synthetic heatwave-afternoon forcing, expect the low-SVF built classes (LCZ 1, 2) to land in a stronger heat-stress category than the open and natural classes (LCZ 9, 14, 17) — even though every point received identical air temperature, wind, and humidity input. This is precisely the kind of differentiated, neighborhood-level heat-health signal that a single city-wide weather-station reading cannot provide on its own, and that motivates pairing UTCI with an LCZ map in heat-health early-warning applications.

## Conclusion

This notebook covered the two purely array-based thermal functions in `lcz4r_python`: `lcz_utci`, which turns air temperature, wind, humidity, and an LCZ-derived mean radiant temperature into a standardized human heat-stress index and category; and `lcz_anthropogenic_heat`, which estimates the waste-heat flux (Q_F) that dense urban LCZs themselves add back into the local energy balance, via either a fast literature-based lookup (`"simple"`) or a morphology-driven parameterization (`"detailed"`). Together with the surface/canopy analyses from earlier notebooks, these tools connect LCZ classification directly to actionable human-health and energy-balance quantities.

**Previous**: [`06_temporal_climate_metrics`](06_temporal_climate_metrics.en.ipynb) — diurnal temperature range and degree-hours by LCZ class.

**Next**: [`08_drought_indices_spi_spei`](08_drought_indices_spi_spei.en.ipynb) — precipitation-based drought indices (SPI/SPEI) computed per municipality.